# Exercise 6 — Bayes Classification and Naive Bayes

In this notebook, you will learn:

* what Bayes classification means
* how probabilities can be used for classification
* why Naive Bayes is useful for text
* how to build a simple spam / not spam classifier
* how NLP turns text into numbers

## Part A — Classification with Probabilities

In classification, we want to answer: `Which class is most likely?`

Example:
`Email → spam or not spam`

Bayes classification chooses the class with the highest probability: `choose the class with the highest P(class | data)`


## Part B — Bayes Rule

Bayes rule is: `P(class | data) = P(data | class) × P(class) / P(data)`

For classification, we compare classes.

Since P(data) is the same for all classes, we can compare: `score = P(data | class) × P(class)`
The class with the highest score wins.

## Part C — Intuition

Imagine we receive this message: `"win money now"`

Is this more likely to be spam or not spam?

Naive Bayes checks:

* how common spam is
* how common these words are in spam
* how common these words are in normal messages

## Part D — Tiny Training Dataset

<img src="https://c4.wallpaperflare.com/wallpaper/701/872/935/mr-robot-tv-series-hello-friend-elliot-mr-robot-hd-wallpaper-preview.jpg" width="250">

In [ ]:
import pandas as pd

data = [
    {"message": "hello friend", "label": "normal"},
    {"message": "hello book", "label": "normal"},
    {"message": "friend book", "label": "normal"},
    {"message": "hello friend book money", "label": "normal"},
    {"message": "hello friend", "label": "normal"},
    {"message": "book friend", "label": "normal"},

    {"message": "win money", "label": "spam"},
    {"message": "money now", "label": "spam"},
    {"message": "win money now", "label": "spam"},
]

df = pd.DataFrame(data)
df

## Part E — Gentle NLP Introduction

Machine learning models cannot read raw text directly.

We need to transform text into numbers.

This process is called: `text vectorization`

A very simple method is: `count the words`

Example: `"hello friend money"`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texts = ["hello friend money money!"]

# Create the vectorizer using Scikit-learn
vectorizer = CountVectorizer()

# Transform text into numbers
X = vectorizer.fit_transform(texts)


In [ ]:
# Show the words
vectorizer.get_feature_names_out()


In [ ]:
# Show the numeric vector
X.toarray()

## Part F — Vocabulary

A vocabulary is the list of words the model knows.

In [ ]:
X_counts = vectorizer.fit_transform(df["message"])

vocabulary = vectorizer.get_feature_names_out()

print(vocabulary)

## Part G — Word Count Table

In [ ]:
X_table = pd.DataFrame(
    X_counts.toarray(),
    columns=vocabulary
)

X_table["label"] = df["label"]

X_table

### Questions

1. What does one row represent?
2. What does one column represent?
3. Why do we need to convert text into numbers?
4. Which words seem associated with spam?
5. Which words seem associated with normal messages?

## Part H — Priors

The prior probability is how common each class is before seeing the message.

In [ ]:
class_counts = df["label"].value_counts()
priors = class_counts / len(df)

priors


* `normal is more common because we have more normal messages`
* `spam is less common because we have fewer spam messages`

### Word Probabilities by Class

Now we count how often each word appears in each class.

In [ ]:
word_counts_by_class = X_table.groupby("label")[vocabulary].sum()

word_counts_by_class

### Convert Counts to Probabilities

This divides each word count by the total number of words in that class.

In [ ]:
word_probs_by_class = word_counts_by_class.div(
    word_counts_by_class.sum(axis=1),
    axis=0
)

word_probs_by_class

### Questions

1. Which words have high probability in spam?
2. Which words have high probability in normal messages?
3. What happens if a word never appears in one class?
4. Why might this be a problem?

## Part J — Classify a New Message Manually

New message: `"hello money"`
We compare:

`score(normal) = P(normal) × P(hello | normal) × P(money | normal)`

`score(spam) = P(spam) × P(hello | spam) × P(money | spam)`


In [ ]:
new_message = "money money"

words = new_message.split()

scores = {}

for label in ["normal", "spam"]:
    score = priors[label]

    for word in words:
        score = score * word_probs_by_class.loc[label, word]

    scores[label] = score

scores

### Decision

In [ ]:
predicted_class = max(scores, key=scores.get)

print("Predicted class:", predicted_class)

### Part K — Problem: Zero Probabilities

If a word never appears in a class, its probability is zero.

Example: `P(book | spam) = 0`

Then the whole score becomes zero: `P(spam) × P(book | spam) × ...`

This is too harsh.
    

## Part L — Laplace Smoothing

Laplace smoothing fixes zero probabilities by adding 1 to every word count. `smoothed count = count + 1`

This means: `No word gets probability zero`

#### Apply Laplace Smoothing

In [ ]:
alpha = 1

smoothed_counts = word_counts_by_class + alpha

smoothed_probs = smoothed_counts.div(
    smoothed_counts.sum(axis=1),
    axis=0
)

smoothed_probs

#### Classify Again With Smoothing

In [ ]:
new_message = "money money"

words = new_message.split()

scores = {}

for label in ["normal", "spam"]:
    score = priors[label]

    for word in words:
        score = score * smoothed_probs.loc[label, word]

    scores[label] = score

scores

In [ ]:
predicted_class = max(scores, key=scores.get)

print("Predicted class:", predicted_class)

### Questions

1. Did smoothing change the probabilities?
2. Why is smoothing useful?
3. Why should unseen words not make the whole score zero?

## Part M — Use Scikit-Learn Naive Bayes

Now we use a ready-made Naive Bayes classifier.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

X = X_counts
y = df["label"]

model = MultinomialNB()

model.fit(X, y)

#### Predict New Messages

In [ ]:
new_messages = [
    "hello friend",
    "win money now",
    "book money",
    "hello book",
]

X_new = vectorizer.transform(new_messages)

predictions = model.predict(X_new)

for message, pred in zip(new_messages, predictions):
    print(message, "→", pred)

#### Prediction Probabilities

In [ ]:
probabilities = model.predict_proba(X_new)

pd.DataFrame(
    probabilities,
    columns=model.classes_,
    index=new_messages
)

## Part N — Evaluate the Model

Since our tiny dataset is very small, evaluation is limited.

But we can still inspect predictions.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X)

accuracy = accuracy_score(y, y_pred)

print("Accuracy:", accuracy)

#### Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y, y_pred, labels=["normal", "spam"])

plt.figure(figsize=(5, 5))
plt.imshow(cm, cmap="Blues")

plt.xticks([0, 1], ["normal", "spam"])
plt.yticks([0, 1], ["normal", "spam"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=18)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")

plt.colorbar()
plt.show()

## Part O — Try a Real SMS Spam Dataset

For a real dataset, use the UCI SMS Spam Collection:

UCI SMS Spam Collection￼ https://archive.ics.uci.edu/dataset/228/sms+spam+collection

A convenient raw version:

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

sms_df = pd.read_csv(
    url,
    sep="\\t",
    header=None,
    names=["label", "message"]
)

sms_df.head()

#### Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    sms_df["message"],
    sms_df["label"],
    test_size=0.25,
    random_state=42
)

#### Vectorize Text

In [ ]:
vectorizer = CountVectorizer(stop_words="english")

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

#### Train Naive Bayes

In [ ]:
model = MultinomialNB()

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label="spam"))
print("Recall:", recall_score(y_test, y_pred, pos_label="spam"))
print("F1:", f1_score(y_test, y_pred, pos_label="spam"))

#### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])

plt.figure(figsize=(5, 5))
plt.imshow(cm, cmap="Blues")

plt.xticks([0, 1], ["ham", "spam"])
plt.yticks([0, 1], ["ham", "spam"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=18)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("SMS Spam Confusion Matrix")

plt.colorbar()
plt.show()

#### Error Analysis

In [ ]:
results = pd.DataFrame({
    "message": X_test_text,
    "true": y_test,
    "predicted": y_pred
})

mistakes = results[results["true"] != results["predicted"]]

mistakes.head(10)

### Questions

1. What kinds of messages are misclassified?
2. Are false positives or false negatives more problematic here?
3. Which words seem to influence the model?
4. Does removing stopwords help?
5. Would a human make the same mistakes?

## Extra Exercise — Use TF-IDF Instead of Word Counts

Until now, we used `CountVectorizer`, which counts how many times each word appears.

Now try using TF-IDF with `TfidfVectorizer` instead. Docs [here](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

**Task:**

Replace this code:

```python
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english")

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: import the TF-IDF vectorizer

# TODO: create the vectorizer

# TODO: fit on the training text and transform it

# TODO: transform the test text

### To remember

* Bayes classification uses probabilities to choose the most likely class.
* Naive Bayes assumes words/features are independent given the class.
* Text must be converted into numbers before machine learning.
* Naive Bayes is **simple, fast, and often strong** for text classification.